# IMOS realtime data to DwC Event Core - SMRU Example

Plan: Convert the realtime QCed IMOS marine mammal position data to DwC, and then publish the result to the IPT.

Contemporary notes from our meet w/ Ian Jonsen here: https://docs.google.com/document/d/1hibIxBbyGwa7b5-LRpnKIyjnr41EkUPKBzdUJoyAfaU/edit#heading=h.6bqw4binj5hq

### Inputs / configuration parameters:

* QCed data for a given campaign or project as the exported CSVs with appended position correction data as per https://github.com/ianjonsen/ArgosQC
* credentials for IPT and 
* corresponding project ID to associate new data with
* minimum quality hit to keep

In [22]:
import pandas as pd

metadata_df = pd.read_csv('./input/imos_ct182/aodn/metadata_ct182_nrt.csv')
loc_df = pd.read_csv('./input/imos_ct182/aodn/diag_ct182_nrt.csv')

In [23]:
metadata_df[0:10]

,sattag_program,device_id,ptt,body,device_wmo_ref,tag_type,common_name,species,release_longitude,release_latitude,...,release_date,recovery_date,age_class,sex,length,estimated_mass,actual_mass,state_country,qc_start_date,qc_end_date
0,ct182,ct182-05-23,255049,15853,Q9902053,CTD_CONT_23BSD,southern elephant seal,Mirounga leonina,70.193679,-49.348771,...,2024-12-24T04:00:22Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-18T14:00:00Z,2025-04-14T14:00:00Z
1,ct182,ct182-865F-24,265866,15865,Q9902084,FTD_GEN_24L,southern elephant seal,Mirounga leonina,70.245170,-49.366088,...,2025-01-07T20:10:13Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-24T11:00:00Z,2025-07-24T11:00:00Z
2,ct182,ct182-866F-24,265867,15866,Q9902085,FTD_GEN_24L,southern elephant seal,Mirounga leonina,70.336980,-49.497705,...,2025-01-07T18:03:18Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-08T22:00:00Z,2025-02-05T10:00:00Z
3,ct182,ct182-883C-24,255061,15883,Q9902086,CTD_GEN_24A,southern elephant seal,Mirounga leonina,70.212102,-49.353200,...,2024-12-23T18:17:51Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-10T20:00:00Z,2025-03-06T14:00:00Z
4,ct182,ct182-884C-24,255058,15884,Q9902087,CTD_GEN_24A,southern elephant seal,Mirounga leonina,72.594369,-48.912253,...,2024-12-23T16:35:48Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2024-12-27T00:00:00Z,2025-07-14T12:00:00Z
5,ct182,ct182-885C-24,255057,15885,Q9902088,CTD_GEN_24A,southern elephant seal,Mirounga leonina,70.200098,-49.354042,...,2025-01-07T23:53:07Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-28T23:00:00Z,2025-05-13T05:00:00Z
6,ct182,ct182-886C-24,255066,15886,Q9902089,CTD_GEN_24A,southern elephant seal,Mirounga leonina,70.227514,-49.353272,...,2025-01-07T00:18:21Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-09T16:00:00Z,2025-07-23T22:00:00Z
7,ct182,ct182-887C-24,255063,15887,Q9902090,CTD_GEN_24A,southern elephant seal,Mirounga leonina,70.247734,-49.377458,...,2025-01-06T20:22:22Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-17T07:00:00Z,2025-07-24T19:00:00Z
8,ct182,ct182-888C-24,255062,15888,Q9902091,CTD_GEN_24A,southern elephant seal,Mirounga leonina,71.653354,-49.016840,...,2025-02-04T12:31:51Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-02-04T12:00:00Z,2025-03-01T12:00:00Z
9,ct182,ct182-890C-I2C-24,255068,15890,Q9902092,CTD_GEN_24A,southern elephant seal,Mirounga leonina,71.326645,-49.423049,...,2025-01-22T16:56:30Z,NaN,NaN,NaN,NaN,NaN,NaN,French Overseas Territory,2025-01-22T21:00:00Z,2025-07-24T03:00:00Z


## Metadata - create events and occurrences for each row

Process the metadata csv into Event Core (animal releases + tag attachments) + Occurrences (HumanObservations) + emofs for same (biological measurements are here)

In [24]:
metadata_df

# event entries: eventID = [body]-[release_date]
#                eventDate =  [release_date]
#                latitude = [release_latitude]
#                longitude = [release_longitude]
#                modified = current_date()
#                geodeticDatum = EPSG:4326
#                country = state_country  (error in current dataset, French Overseas Territory should be French Southern Lands)

column_map = {'release_date':'eventDate',
              'release_latitude':'decimalLatitude',
              'release_longitude':'decimalLongitude',
              'state_country':'country'}

event_df = metadata_df.rename(columns=column_map)
event_df['modified'] = pd.to_datetime('now', utc=True).round(freq='s')
# eventID is instrument serial number (body) + release datetime (eventDate)
event_df['eventID'] = event_df['body'].astype(str).str.cat(event_df['eventDate'].astype(str), sep='-')
event_df['geodeticDatum'] = 'EPSG:4326'
# Optional: truncate the extra columns from the core
event_df =  event_df[['eventID', 'eventDate', 'decimalLatitude', 'decimalLongitude', 'modified', 'geodeticDatum', 'country']]

In [25]:
event_df[0:5]

,eventID,eventDate,decimalLatitude,decimalLongitude,modified,geodeticDatum,country
0,15853-2024-12-24T04:00:22Z,2024-12-24T04:00:22Z,-49.348771,70.193679,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory
1,15865-2025-01-07T20:10:13Z,2025-01-07T20:10:13Z,-49.366088,70.245170,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory
2,15866-2025-01-07T18:03:18Z,2025-01-07T18:03:18Z,-49.497705,70.336980,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory
3,15883-2024-12-23T18:17:51Z,2024-12-23T18:17:51Z,-49.353200,70.212102,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory
4,15884-2024-12-23T16:35:48Z,2024-12-23T16:35:48Z,-48.912253,72.594369,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory


In [26]:
# EMOFs to harvest
# for the release events
# instrument manufacturer and model  (SMRU + [tag type])
# PTT
# device id
# WMO ref




In [27]:
# occ ext. entries:    occurrenceID = [body]-[release_date]
#                      species = [species]
#                      sex = [sex]
#                      eventID = [body]-[release_date]
#                      organismID = [body]-[release_date]

occ_column_map = {'release_date':'eventDate',
                  'species':'scientificName'}
occ_df = metadata_df.rename(columns=occ_column_map)
occ_df['occurrenceID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['eventID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['organismID'] = occ_df['body'].astype(str).str.cat(occ_df['eventDate'].astype(str), sep='-')
occ_df['basisOfRecord'] = 'HumanObservation'
occ_df = occ_df[['occurrenceID', 'organismID','eventID', 'sex', 'scientificName', 'basisOfRecord']]

In [28]:
occ_df[0:5]

,occurrenceID,organismID,eventID,sex,scientificName,basisOfRecord
0,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z,NaN,Mirounga leonina,HumanObservation
1,15865-2025-01-07T20:10:13Z,15865-2025-01-07T20:10:13Z,15865-2025-01-07T20:10:13Z,NaN,Mirounga leonina,HumanObservation
2,15866-2025-01-07T18:03:18Z,15866-2025-01-07T18:03:18Z,15866-2025-01-07T18:03:18Z,NaN,Mirounga leonina,HumanObservation
3,15883-2024-12-23T18:17:51Z,15883-2024-12-23T18:17:51Z,15883-2024-12-23T18:17:51Z,NaN,Mirounga leonina,HumanObservation
4,15884-2024-12-23T16:35:48Z,15884-2024-12-23T16:35:48Z,15884-2024-12-23T16:35:48Z,NaN,Mirounga leonina,HumanObservation


In [29]:
# EMOFs to harvest
# for the occurrences:
# sex
# length
# weight

In [30]:
# Create event and occurrence entries from the locations data file
# 
# Event entries:  eventID = organismID + date_detected
#                 latitude = ssm_lat if exists else lat
#                 longitude = ssm_lon if exists else lon
#                 eventDate = d_date
#                 geodeticDatum = EPSG:4326
#                 coordinateUncertaintyInMeters = max(ssm_x_se, ssm_y_se)  -- 1 SE or 2 SE?
#                 

# add the relevant columns to loc_df from metadata_df to create organismID
loc_df = loc_df.merge(metadata_df[['device_id', 'body', 'release_date', 'species']], 
                      how='left', left_on='ref', right_on='device_id')


In [31]:
# combine the organismID + the detection date into the eventID
loc_df['eventID'] = loc_df['body'].astype(str).str.cat(loc_df[['release_date', 'd_date']], sep='-')

In [32]:

# Check: is this correct to do in all cases?
# where there has been no correction made (corrected positions = NA, 
#       then use the raw position data
loc_df['decimalLatitude'] = loc_df['ssm_lat'].fillna(loc_df['lat'])
loc_df['decimalLongitude'] = loc_df['ssm_lon'].fillna(loc_df['lon'])
loc_df['eventDate'] = loc_df['d_date']
loc_df['modified'] = pd.to_datetime('now', utc=True).round(freq='s') 

# constant
loc_df['geodeticDatum'] = 'EPSG:4326'

# Ian's got his ssm_x and ssm_y in km, not in m
loc_df['coordinateUncertaintyInMeters'] = loc_df[['ssm_x_se', 'ssm_y_se']].max(axis=1) * 1000

# revisit uncertainty - make a radius based on the max, but uncertainty is an ellipse
# OBIS doesn't know about it but we can make a Polygon and include it somewhere to preserve the better
# knowledge that we have.

# Where there are multiple hits for a given time step (many satellites have opinions on position at once), 

loc_df = loc_df.sort_values(['ref', 'd_date', 'lq'], ascending=False)
# drop all but the best of the location qualities
loc_df = loc_df.drop_duplicates(subset=['ref','d_date'], keep='first', inplace=False)


In [33]:
loc_df['lq'].unique()

array([-2,  3,  2,  1, -1,  0, -9])

In [34]:
# fallback - where coordinateUncertaintyInMeters is still null (un-QCed), 
# let's do something with the class of fix from Argos. set a lookup table as in the ATN example?

# Ian recommends - dan costa, accuracy of argos locations at sea pinnipeds
# in that article they compared GPS to Argos locations.
# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0008677

# While this paper recommends different regimes per species due to differences in surfacing behaviour, 
# we don't have that kind of broad data in real-time land.

# So we take their recommendations for marine mammals here, and maybe we'd leave the door open to use a non-mammal error chart
# for non-airbreathers

# Other methodologies have thrown out the A and B quality hits altogther. 
# I'm not opposed to doing that but i'll confirm it with the SME beforehand
# because the QC algorithm is re-positioning the bad hits for us already.
# and if they didn't throw them out, we may not want to either.

# 68th percentile location error distances from Costa et al, in metres
# old LQ designations - CLS moved to a Kalman-filtered location set in ~2011
# error ellipses are now de riguer - semimajor and 
# semiminor ellipse axis/orientation to quantify uncertainty
# So we could harvest those first.

error_table = {3:490,
               2:1010,
               1:1200,
               0:4180,
               -1:6190,
               -2:10280,
               -9:10280} # TODO : What is the corresponding code to -9 LQ? 
                         # AniMotum thinks it's a class B
missing_errors = loc_df['coordinateUncertaintyInMeters'].isna()
loc_df.loc[missing_errors, 'coordinateUncertaintyInMeters'] = loc_df.loc[missing_errors, 'lq'].map(error_table)

In [35]:
loc_df['coordinateUncertaintyInMeters'].describe()

count    7.582800e+04
mean     2.963805e+03
std      1.431511e+04
min      1.455000e+01
25%      1.465237e+03
50%      2.160929e+03
75%      3.314880e+03
max      3.845497e+06
Name: coordinateUncertaintyInMeters, dtype: float64

In [36]:
loc_df['modified']

91246   2025-09-05 16:38:39+00:00
91245   2025-09-05 16:38:39+00:00
91244   2025-09-05 16:38:39+00:00
91243   2025-09-05 16:38:39+00:00
91242   2025-09-05 16:38:39+00:00
                   ...           
4       2025-09-05 16:38:39+00:00
3       2025-09-05 16:38:39+00:00
2       2025-09-05 16:38:39+00:00
1       2025-09-05 16:38:39+00:00
0       2025-09-05 16:38:39+00:00
Name: modified, Length: 75828, dtype: datetime64[ns, UTC]

In [37]:
# Select the columns and append to the event_df

event_df = pd.concat([event_df, loc_df[['eventID', 'eventDate', 'decimalLatitude', 'decimalLongitude', 'modified','geodeticDatum', 'coordinateUncertaintyInMeters']]])

In [38]:
event_df

,eventID,eventDate,decimalLatitude,decimalLongitude,modified,geodeticDatum,country,coordinateUncertaintyInMeters
0,15853-2024-12-24T04:00:22Z,2024-12-24T04:00:22Z,-49.348771,70.193679,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory,NaN
1,15865-2025-01-07T20:10:13Z,2025-01-07T20:10:13Z,-49.366088,70.245170,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory,NaN
2,15866-2025-01-07T18:03:18Z,2025-01-07T18:03:18Z,-49.497705,70.336980,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory,NaN
3,15883-2024-12-23T18:17:51Z,2024-12-23T18:17:51Z,-49.353200,70.212102,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory,NaN
4,15884-2024-12-23T16:35:48Z,2024-12-23T16:35:48Z,-48.912253,72.594369,2025-09-05 16:38:35+00:00,EPSG:4326,French Overseas Territory,NaN
...,...,...,...,...,...,...,...,...
4,15853-2024-12-24T04:00:22Z-2024-12-24T05:20:53Z,2024-12-24T05:20:53Z,-49.350350,70.196410,2025-09-05 16:38:39+00:00,EPSG:4326,NaN,490.0
3,15853-2024-12-24T04:00:22Z-2024-12-24T04:40:38Z,2024-12-24T04:40:38Z,-49.349100,70.196180,2025-09-05 16:38:39+00:00,EPSG:4326,NaN,490.0
2,15853-2024-12-24T04:00:22Z-2024-12-24T04:39:10Z,2024-12-24T04:39:10Z,-49.348980,70.194700,2025-09-05 16:38:39+00:00,EPSG:4326,NaN,490.0
1,15853-2024-12-24T04:00:22Z-2024-12-24T04:11:28Z,2024-12-24T04:11:28Z,-49.348510,70.182710,2025-09-05 16:38:39+00:00,EPSG:4326,NaN,1200.0


In [39]:
# Occurrence entries: occurrenceID = eventID
#                     eventID = eventID
#                     species = species
#                     organismID = body + release_date

loc_df['occurrenceID'] = loc_df['eventID']
loc_df['organismID'] =  loc_df['body'].astype(str).str.cat(loc_df['release_date'].astype(str), sep='-') 

In [40]:
# Decimate to first each hour per animal. Acoustics would also use per-receiver location, argos and sat won't need that.
dets_df = loc_df
dets_df['scientificName'] = dets_df['species']
dets_df['basisOfRecord'] = 'MachineObservation'
dets_df['Date'] = pd.to_datetime(dets_df['d_date']).dt.date
dets_df['hr'] = pd.to_datetime(dets_df['d_date']).dt.hour
dets_df['binsize'] = dets_df.groupby(['organismID', 'Date', 'hr']).size().reset_index(name='binsize')['binsize']
dets_df.drop_duplicates(subset=['organismID','Date', 'hr'], keep='first', inplace=True)
dets_df.drop('hr', axis=1, inplace=True)
dets_df

,ref,ptt,d_date,lq,lat,lon,alt_lat,alt_lon,n_mess,n_mess_120,...,eventDate,modified,geodeticDatum,coordinateUncertaintyInMeters,occurrenceID,organismID,scientificName,basisOfRecord,Date,binsize
91246,ct182-970F-24,265875,2025-05-09T10:31:48Z,-2,-49.35761,70.20780,-49.35761,70.20780,2,0,...,2025-05-09T10:31:48Z,2025-09-05 16:38:39+00:00,EPSG:4326,10280.0,15970-2025-01-07T19:44:13Z-2025-05-09T10:31:48Z,15970-2025-01-07T19:44:13Z,Mirounga leonina,MachineObservation,2025-05-09,NaN
91245,ct182-970F-24,265875,2025-05-09T05:27:59Z,3,-49.35058,70.24171,-49.35058,70.24171,13,2,...,2025-05-09T05:27:59Z,2025-09-05 16:38:39+00:00,EPSG:4326,490.0,15970-2025-01-07T19:44:13Z-2025-05-09T05:27:59Z,15970-2025-01-07T19:44:13Z,Mirounga leonina,MachineObservation,2025-05-09,NaN
91244,ct182-970F-24,265875,2025-05-09T00:15:52Z,2,-49.34430,70.20447,-49.34430,70.20447,9,0,...,2025-05-09T00:15:52Z,2025-09-05 16:38:39+00:00,EPSG:4326,1010.0,15970-2025-01-07T19:44:13Z-2025-05-09T00:15:52Z,15970-2025-01-07T19:44:13Z,Mirounga leonina,MachineObservation,2025-05-09,NaN
91243,ct182-970F-24,265875,2025-05-08T14:49:09Z,2,-49.35236,70.22635,-49.35236,70.22635,9,0,...,2025-05-08T14:49:09Z,2025-09-05 16:38:39+00:00,EPSG:4326,1010.0,15970-2025-01-07T19:44:13Z-2025-05-08T14:49:09Z,15970-2025-01-07T19:44:13Z,Mirounga leonina,MachineObservation,2025-05-08,NaN
91240,ct182-970F-24,265875,2025-05-08T04:21:31Z,3,-49.36614,70.24162,-49.36614,70.24162,13,0,...,2025-05-08T04:21:31Z,2025-09-05 16:38:39+00:00,EPSG:4326,490.0,15970-2025-01-07T19:44:13Z-2025-05-08T04:21:31Z,15970-2025-01-07T19:44:13Z,Mirounga leonina,MachineObservation,2025-05-08,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,ct182-05-23,255049,2024-12-24T13:13:40Z,3,-49.34945,70.19620,-49.34945,70.19620,7,1,...,2024-12-24T13:13:40Z,2025-09-05 16:38:39+00:00,EPSG:4326,490.0,15853-2024-12-24T04:00:22Z-2024-12-24T13:13:40Z,15853-2024-12-24T04:00:22Z,Mirounga leonina,MachineObservation,2024-12-24,2.0
9,ct182-05-23,255049,2024-12-24T12:40:37Z,2,-49.34721,70.18504,-49.34721,70.18504,11,1,...,2024-12-24T12:40:37Z,2025-09-05 16:38:39+00:00,EPSG:4326,1010.0,15853-2024-12-24T04:00:22Z-2024-12-24T12:40:37Z,15853-2024-12-24T04:00:22Z,Mirounga leonina,MachineObservation,2024-12-24,1.0
8,ct182-05-23,255049,2024-12-24T06:55:24Z,-1,-49.34810,70.19144,-49.34810,70.19144,3,0,...,2024-12-24T06:55:24Z,2025-09-05 16:38:39+00:00,EPSG:4326,6190.0,15853-2024-12-24T04:00:22Z-2024-12-24T06:55:24Z,15853-2024-12-24T04:00:22Z,Mirounga leonina,MachineObservation,2024-12-24,3.0
6,ct182-05-23,255049,2024-12-24T05:51:52Z,2,-49.35083,70.19323,-49.35083,70.19323,6,0,...,2024-12-24T05:51:52Z,2025-09-05 16:38:39+00:00,EPSG:4326,1010.0,15853-2024-12-24T04:00:22Z-2024-12-24T05:51:52Z,15853-2024-12-24T04:00:22Z,Mirounga leonina,MachineObservation,2024-12-24,3.0


In [41]:
dets_df['binsize'].describe()

count    23306.000000
mean         1.611216
std          0.852076
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: binsize, dtype: float64

In [42]:
dets_df['dataGeneralizations'] = dets_df['binsize'].apply(lambda x: 'subsampled by hour, first of {} record(s)'.format(x))

In [43]:
occ_df = pd.concat([occ_df, dets_df[['occurrenceID', 'eventID', 'scientificName', 'organismID', 'basisOfRecord']]])

In [44]:
# flesh out the occurrence taxonomic entries with kingdom, phylum, class, order, family
import pyworms

lookup_dict = {}
for name in occ_df['scientificName'].unique():
    resp = pyworms.aphiaRecordsByMatchNames(name)
    if len(resp[0]) == 0:
        print('\nNo match for name "{}"'.format(name))
        continue
    elif len(resp[0]) > 1:
        print('\nMultiple matches for name "{}"'.format(name))
        pprint.pprint(resp[0], indent=4)
        continue
    else:
        worms = resp[0][0]
        lookup_dict[name]={'scientificName': name,
                           'scientificNameID': worms['lsid'],
                           'taxonRank': worms['rank'],
                           'kingdom': worms['kingdom'],
                           'phylum': worms['phylum'],
                           'class': worms['class'],
                           'order': worms['order'],
                           'family': worms['family']}
        
lookup_df = pd.DataFrame.from_dict(lookup_dict, orient='index')

In [45]:
lookup_df

,scientificName,scientificNameID,taxonRank,kingdom,phylum,class,order,family
Mirounga leonina,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae


In [46]:
occ_df = occ_df.join(lookup_df, how='left', on='scientificName', rsuffix='_worms')

In [47]:
occ_df

,occurrenceID,organismID,eventID,sex,scientificName,basisOfRecord,scientificName_worms,scientificNameID,taxonRank,kingdom,phylum,class,order,family
0,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z,NaN,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
1,15865-2025-01-07T20:10:13Z,15865-2025-01-07T20:10:13Z,15865-2025-01-07T20:10:13Z,NaN,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
2,15866-2025-01-07T18:03:18Z,15866-2025-01-07T18:03:18Z,15866-2025-01-07T18:03:18Z,NaN,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
3,15883-2024-12-23T18:17:51Z,15883-2024-12-23T18:17:51Z,15883-2024-12-23T18:17:51Z,NaN,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
4,15884-2024-12-23T16:35:48Z,15884-2024-12-23T16:35:48Z,15884-2024-12-23T16:35:48Z,NaN,Mirounga leonina,HumanObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,15853-2024-12-24T04:00:22Z-2024-12-24T13:13:40Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z-2024-12-24T13:13:40Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
9,15853-2024-12-24T04:00:22Z-2024-12-24T12:40:37Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z-2024-12-24T12:40:37Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
8,15853-2024-12-24T04:00:22Z-2024-12-24T06:55:24Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z-2024-12-24T06:55:24Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae
6,15853-2024-12-24T04:00:22Z-2024-12-24T05:51:52Z,15853-2024-12-24T04:00:22Z,15853-2024-12-24T04:00:22Z-2024-12-24T05:51:52Z,NaN,Mirounga leonina,MachineObservation,Mirounga leonina,urn:lsid:marinespecies.org:taxname:231413,Species,Animalia,Chordata,Mammalia,Carnivora,Phocidae


In [48]:
occ_df['organismID'].unique()

array(['15853-2024-12-24T04:00:22Z', '15865-2025-01-07T20:10:13Z',
       '15866-2025-01-07T18:03:18Z', '15883-2024-12-23T18:17:51Z',
       '15884-2024-12-23T16:35:48Z', '15885-2025-01-07T23:53:07Z',
       '15886-2025-01-07T00:18:21Z', '15887-2025-01-06T20:22:22Z',
       '15888-2025-02-04T12:31:51Z', '15890-2025-01-22T16:56:30Z',
       '15891-2025-01-21T11:11:42Z', '15892-2025-01-28T08:21:17Z',
       '15893-2025-01-21T01:29:37Z', '15894-2025-01-08T01:49:10Z',
       '15895-2025-01-19T23:09:57Z', '15940-2025-02-02T07:51:54Z',
       '15949-2025-01-26T05:55:13Z', '15952-2024-12-24T11:03:32Z',
       '15953-2024-12-23T07:43:45Z', '15954-2025-01-07T23:53:54Z',
       '15955-2025-01-09T19:57:16Z', '15959-2025-01-08T20:23:26Z',
       '15964-2025-01-07T18:13:25Z', '15966-2025-01-07T20:36:49Z',
       '15967-2025-01-08T14:28:35Z', '15969-2024-12-23T11:32:49Z',
       '15970-2025-01-07T19:44:13Z'], dtype=object)

In [49]:
# Any EMOFs to harvest from detection occurrences?
# 
# 

In [50]:
# EMOFs from instruments onboard 

# CTD - in the ctd_xxx.csv
# end_date - the time the tag broke the surface after the dive (post-measurements in the current row)
# position supplied by Ian's code is keying off of end_date, not any of the created/updated datetimes in the row.

# row has multiple depth/measurement/corrected measurement rows 
# within that correspond to previous measurements along the last dive, curated by the tag
# Profiles arrive in near realtime but were collected some-time (maybe as many as a few days) in the past.
# Tag processes how many deep-dives made in last 24 hours, which are complete, 
# was it enough for a CTD profile, yes, then transmit
# Was the transmission received and arrived at SMRU? Yes, it will be in this table.

ctd_df = pd.read_csv('input/imos_ct182/aodn/ctd_ct182_nrt.csv')
#ctd_df

dive_df = pd.read_csv('input/imos_ct182/aodn/dive_ct182_nrt.csv')
ctd_with_dives_df = dive_df.merge(ctd_df, how='left', left_on=['ref','ptt', 'de_date'], right_on=['ref', 'ptt', 'end_date'])

# Shape of a dive not knowable, but all depths are from after the previous message and up to the current end_date.
# e-2           e-1  e
# |-------------|----|

# Dive table has all recorded dives, and the start/end time of those dives.
# Search for the CTD dives in the dives table, figure out when dives start and end. ds_date and de_date in dive table

# What if we: join dive table to ctd table, get ds_date and de_date - > map to eventDate for a full dive.
# make 'dive' events with no Occurrences, assign the measurements to the dive Event.
# or associate all emofs to the dive record?

# eventDate ds_date/de_date
# 

In [70]:
# check the merged data out
# some tags will be fluorescence tags - will get these but also a fluorescence value.
# fluor_vals
# where ### is the campaign IDs.
# CT tags have ct###_xxx for their ref
# FT tags have ft###_xxx for their ref

# Generalized:
# For each instrument_dbar pair
# Check depth column for nans
# for all non-nans
# Create if it isn't already in the event table - a new event for this measurement/dive
# Create emofs associated with that event, at each of the depths for the non-nan measurement.
# Need: list of instrument value columns. temp_ sal_ cond_ fluoro_, etc. It's *_dbar and *_vals
# There's a pair of [inst]_dbar / [inst]_vals columns per measurement.
# Check - dbar follows instrument packages (ctd and cond often combined, but oxy / fluo separate) so use 
# inst_dbar for each inst_vals.
# x, y and _se vals are in km - SSMs would throw a fit if we had things in metres.
# projections that the model run used to make x,y is documented in the project config file. 
# for ct182 here, it's Stereographic centred on where Macquarie Island is.

ctd_with_dives_df[~ctd_with_dives_df['temp_dbar'].isna()][['ref', 'ptt', #identify individuals
                                                           'ds_date', 
                                                           'de_date', 
                                                           'temp_dbar', 'temp_vals',
                                                           'sal_vals',
                                                           'ssm_lon_y','ssm_lat_y','ssm_x_y','ssm_y_y','ssm_x_se_y','ssm_y_se_y']]

,ref,ptt,ds_date,de_date,temp_dbar,temp_vals,sal_vals,ssm_lon_y,ssm_lat_y,ssm_x_y,ssm_y_y,ssm_x_se_y,ssm_y_se_y
543,ct182-892C-24,255055,2025-02-10T06:51:44Z,2025-02-10T07:10:00Z,"4,10,20,50,96,100,200,240,260,300,370,382,400,...","5.211,5.211,5.202,5.173,4.194,4.166,2.770,2.31...","33.919,33.931,33.884,33.838,33.983,33.960,34.1...",84.935670,-53.269188,-1072.859150,3986.054336,3.083189,2.946268
1831,ct182-892C-24,255055,2025-03-20T20:01:12Z,2025-03-20T20:40:00Z,"4,10,20,50,90,92,96,100,106,134,174,200,300,40...","3.443,3.454,3.454,3.449,3.404,3.328,2.664,2.12...","37.706,37.340,37.128,37.010,37.262,37.272,37.2...",109.622716,-57.576910,604.478090,3565.292094,2.044184,0.891107
1959,ct182-892C-24,255055,2025-03-25T00:41:52Z,2025-03-25T01:10:00Z,"4,10,20,50,78,84,90,100,140,170,200,300,454,50...","3.750,3.743,3.740,3.680,3.193,2.570,1.250,0.82...","35.988,35.988,35.988,35.988,35.988,35.988,35.9...",109.501215,-57.076680,606.629036,3624.601315,2.532158,1.539654
14635,ct182-884C-24,255058,2025-01-29T19:21:44Z,2025-01-29T19:40:00Z,"4,10,20,30,36,50,54,60,62,100,200,204,300,400,...","-0.216,-0.232,-0.321,-0.443,-0.578,-1.103,-1.2...","33.510,33.521,33.588,33.577,33.528,33.596,33.6...",117.075574,-65.310245,799.624847,2603.176918,1.122833,0.738186
16906,ct182-884C-24,255058,2025-03-22T06:44:00Z,2025-03-22T07:10:00Z,"4,10,20,46,50,100,200,300,350,376,400,408,424,...","-1.950,-1.947,-1.943,-1.922,-1.697,-1.758,-1.8...","33.824,33.860,33.834,34.052,34.421,33.992,34.2...",148.078873,-66.518000,1924.343358,1727.895673,2.603298,1.409851
23166,ct182-964C-24,255059,2025-03-15T10:14:24Z,2025-03-15T10:30:00Z,"4,10,20,26,30,50,56,58,64,96,100,150,200,300,3...","-1.850,-1.801,-1.707,-1.605,-1.122,-0.719,-0.8...","30.689,30.703,30.460,33.920,30.615,33.976,33.7...",75.751169,-68.856674,-953.912331,2117.723644,2.510504,2.017216
25881,ct182-964C-24,255059,2025-05-17T01:57:36Z,2025-05-17T02:30:00Z,"4,10,20,30,50,75,100,106,118,138,150,184,200,2...","0.434,0.435,0.464,0.486,0.492,0.532,0.529,0.64...","31.274,31.274,31.274,31.274,31.274,31.274,31.2...",77.243593,-57.387784,-1407.381886,3355.179315,3.324670,1.662446
28341,ct182-940C-24,255060,2025-03-08T16:25:20Z,2025-03-08T16:50:00Z,"10,20,50,100,150,200,300,400","4.030,4.030,4.010,0.961,1.486,1.767,2.074,2.101","33.816,33.815,33.826,33.982,34.156,34.276,34.4...",103.439098,-55.639802,230.643880,3837.938057,1.571742,0.508156
29763,ct182-940C-24,255060,2025-04-17T21:51:12Z,2025-04-17T22:10:00Z,"4,10,20,30,50,75,78,82,88,96,100,130,150,174,2...","0.766,0.768,0.771,0.771,0.774,0.586,0.473,0.47...","33.795,33.795,33.794,33.794,33.794,33.915,33.9...",122.603728,-61.372204,1220.033139,2930.405674,2.272556,1.261515
30843,ct182-940C-24,255060,2025-05-25T00:57:36Z,2025-05-25T01:30:00Z,"4,10,20,50,100,150,158,200,234,270,286,300,320...","2.034,2.034,2.036,2.038,2.042,2.043,1.654,1.87...","33.867,33.869,33.866,33.866,33.864,33.862,34.0...",108.716910,-52.575434,638.255951,4162.803934,1.909264,0.729569


In [69]:
ctd_with_dives_df.columns

Index(['ref', 'ptt', 'cnt', 'de_date', 'surf_dur', 'dive_dur', 'max_dep', 'd1',
       'd2', 'd3',
       ...
       'photo_vals', 'lat_y', 'lon_y', 'ssm_lon_y', 'ssm_lat_y', 'ssm_x_y',
       'ssm_y_y', 'ssm_x_se_y', 'ssm_y_se_y', 'cid_y'],
      dtype='object', length=147)

In [68]:
ctd_df[['temp_dbar', 'sal_dbar', 'fluoro_dbar', 'temp_vals', 'ssm_lon', 'ssm_lat', 'ssm_x', 'ssm_y', 'ssm_x_se', 'ssm_y_se']]

,temp_dbar,sal_dbar,fluoro_dbar,temp_vals,ssm_lon,ssm_lat,ssm_x,ssm_y,ssm_x_se,ssm_y_se
0,"4,6,8,10,12,14","4,6,8,10,12,14",NaN,"6.379,6.472,6.510,6.510,6.472,6.419",NaN,NaN,NaN,NaN,NaN,NaN
1,"4,6,10,12,14,16,18,20,22,24,26","4,6,10,12,14,16,18,20,22,24,26",NaN,"5.752,5.738,5.738,5.738,5.729,5.706,5.705,5.69...",NaN,NaN,NaN,NaN,NaN,NaN
2,"4,6,10,12,14,16,18,20,22,24,26,30,32,34,42","4,6,10,12,14,16,18,20,22,24,26,30,32,34,42",NaN,"5.578,5.574,5.547,5.527,5.525,5.519,5.508,5.43...",NaN,NaN,NaN,NaN,NaN,NaN
3,"4,10,14,20,30,40,50,52,56,60,64,75,92,96,100,136","4,10,14,20,30,40,50,52,56,60,64,75,92,96,100,136",NaN,"4.584,4.584,4.572,4.570,4.573,4.578,4.579,4.57...",70.518590,-49.734330,-2242.742477,3967.037391,0.997560,0.767824
4,"4,10,20,30,44,50,75,86,96,100,114,132,140,150,...","4,10,20,30,44,50,75,86,96,100,114,132,140,150,...",NaN,"4.376,4.380,4.376,4.367,4.340,4.245,4.162,4.09...",70.619374,-49.961444,-2222.098579,3946.706342,2.157437,2.923913
...,...,...,...,...,...,...,...,...,...,...
17669,"4,10,20,50,96,100,114,128,140,158,200,224,300,...","4,10,20,50,96,100,114,128,140,158,200,224,300,...",NaN,"4.750,4.750,4.750,4.726,4.630,4.542,4.220,3.62...",70.923050,-50.335060,-2178.923362,3918.462006,2.711098,3.056823
17670,NaN,NaN,NaN,NaN,70.923050,-50.335060,-2178.923362,3918.462006,2.711098,3.056823
17671,"4,10,20,30,50,88,96,100,124,132,150,168,200,27...","4,10,20,30,50,88,96,100,124,132,150,168,200,27...",NaN,"4.840,4.835,4.833,4.833,4.823,4.791,4.596,4.46...",NaN,NaN,NaN,NaN,NaN,NaN
17672,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Push them out to files and an archive:

occ_df.to_csv('./output/imos_ct182/occurrences.csv', date_format='%Y-%m-%dT%H:%M:%S')
event_df.to_csv('./output/imos_ct182/events.csv', date_format='%Y-%m-%dT%H:%M:%S')
# emof_df.to_csv('output/emof.csv', date_format='%Y-%m-%dT%H:%M:%S')

NameError: name 'occ_df' is not defined

In [30]:
import 

# Zip and ship to an IPT

# Either via a form fill-in, or via depositing the archive on the IPT's filesystem?

# TODO: Try the form-fill first - use the OTN IPT workflows from ipython-utilities
# import requests  # session with the forms themselves
# import selenium  # or pick-n-click

### Debugging cells:

In [34]:
# throwing out all but max lq will help us de-duplicate these same-time-same-tag hits?
loc_df['lq'].describe()

count    46901.000000
mean        -1.286881
std          1.037948
min         -9.000000
25%         -2.000000
50%         -2.000000
75%         -1.000000
max          3.000000
Name: lq, dtype: float64